# Random-Network Timing Experiment
This Notebook follows the same strict timing boundaries as the standard-network experiment.
1. The density parameter for random networks is the average in-degree $d=|E|/n$. Therefore, the target number of edges for each graph is $\operatorname{round}(nd)$.
2. The main experiment and the $n=1000$ stress test are run separately. The main experiment uses $n\in\{50,100,200,500\}$, while the stress test is conducted separately with $n=1000$.
3. Warm-up is performed separately for each $(G,C,\beta)$ instance, rather than using only the first conditioning set to warm up a given topological order.
4. The final outputs of FGCR and ROS are checked for instance-wise consistency only after all formal timing runs have been completed. The consistency check is excluded from the timed interval.
Timing protocol: FGCR Algorithm1 and Algorithm2 are timed separately and their runtimes are summed; ROS is timed over its complete core algorithm. Each instance is warmed up 3 times and then formally repeated 10 times, with the median runtime reported. The execution order of the two algorithms is alternated. Garbage collection, structural diagnostics, consistency checks, result aggregation, and result saving are all performed outside the timed interval.

In [1]:
from __future__ import annotations
import gc
import json
import math
import random
import time
from dataclasses import asdict, dataclass
from itertools import combinations
from pathlib import Path
from statistics import median
from typing import Hashable, Iterable, Mapping, Sequence

import networkx as nx
import pandas as pd

# Node can be any hashable object; the current random-network experiments use integer nodes 0,1,...,n-1.
Node = Hashable
# Edge is consistently represented as a tuple (source, target).
Edge = tuple[Node, Node]

## 1. Input Generation and Recording Utilities
Each random DAG is generated by first sampling a latent topological order and then drawing, without replacement, $|E|=\operatorname{round}(nd)$ edges from all admissible forward node pairs, where $d=|E|/n$ denotes the average in-degree. Graph generation, conditioning-set sampling, topological-order generation, and input encoding are all performed outside the formal timing interval.


In [2]:
def _validate_dag(G: nx.DiGraph) -> None:
    """Check outside the timed interval whether the input is a simple DAG."""
    # The algorithm only accepts NetworkX directed graph objects.
    # 算法只接受 networkx 的有向图对象。
    if not isinstance(G, nx.DiGraph):
        raise TypeError("G 必须是 networkx.DiGraph。")
    if G.is_multigraph():
        raise TypeError("G 必须是简单有向图，不能是 MultiDiGraph。")
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("输入图 G 必须是 DAG。")


def _encode_nodes(nodes: Iterable[Node]) -> str:
     """Encode a node set deterministically as a JSON string for Excel output."""
    ordered = sorted(nodes, key=str)
    return json.dumps([str(node) for node in ordered], ensure_ascii=False, separators=(",", ":"))


def _encode_order(order: Sequence[Node]) -> str:
    """Encode a complete topological order as a JSON string while preserving its original order."""
    return json.dumps([str(node) for node in order], ensure_ascii=False, separators=(",", ":"))


def random_topological_order(G: nx.DiGraph, seed: int) -> tuple[Node, ...]:
        """Generate a reproducible complete topological order using randomized Kahn's algorithm."""
     # Input validation is performed during topological-order generation and is excluded from algorithm timing.
    _validate_dag(G)
    rng = random.Random(seed)
    indegree = dict(G.in_degree())
    available = sorted((node for node, degree in indegree.items() if degree == 0), key=str)
    order: list[Node] = []

        # At each iteration, randomly select one node from the currently available root nodes.
    while available:
        selected_index = rng.randrange(len(available))
        # Remove the selected node and append it to the topological order.
        selected_node = available.pop(selected_index)
        order.append(selected_node)
        for child in sorted(G.successors(selected_node), key=str):
            indegree[child] -= 1
            if indegree[child] == 0:
                available.append(child)
        # Re-sort the candidate pool at each iteration so that the same seed gives the same result across runs.
        available.sort(key=str)

    # If not all nodes are covered, the input is not a DAG or an implementation error has occurred.
    if len(order) != G.number_of_nodes():
        raise RuntimeError("拓扑排序未覆盖 G 的全部节点。")
    return tuple(order)


def generate_fixed_random_topological_orders(
    G: nx.DiGraph,
    n_orders: int,
    seed_base: int,
    max_attempts: int = 10_000,
) -> list[dict[str, object]]:
    """Generate n_orders distinct fixed random topological orders for a graph."""
    # records stores the order_id, seed, and the topological order itself.
    records: list[dict[str, object]] = []
    seen: set[tuple[Node, ...]] = set()

     # Increment the seed until enough distinct topological orders have been generated.
    for attempt in range(max_attempts):
        beta_seed = seed_base + attempt
        beta = random_topological_order(G, seed=beta_seed)
        # Do not count duplicate topological orders.
        if beta in seen:
            continue
        seen.add(beta)
        records.append({"order_id": len(records) + 1, "beta_seed": beta_seed, "beta": beta})
        if len(records) == n_orders:
            return records
        
    raise RuntimeError(f"{G.name or 'G'} 只生成了 {len(records)} 个不同拓扑序，目标为 {n_orders}。")


def generate_random_dag_exact_edges(
    n: int,
    mean_indegree: float,
    seed: int,
    name: str = "",
) -> nx.DiGraph:
    """Generate a random DAG with exactly round(n*mean_indegree) edges."""
    if n < 2:
        raise ValueError("n 必须至少为2。")
    if mean_indegree <= 0:
        raise ValueError("mean_indegree 必须为正数。")

    # d is defined as the average in-degree |E|/n, so the target number of edges is round(n*d).
    target_edges = int(round(n * mean_indegree))
     # A simple DAG on n nodes can contain at most n(n-1)/2 edges.
    max_edges = n * (n - 1) // 2
    if target_edges > max_edges:
        raise ValueError(f"目标边数 {target_edges} 超过 DAG 最大边数 {max_edges}。")

    rng = random.Random(seed)
    hidden_order = list(range(n))
    rng.shuffle(hidden_order)

     # Sample only forward position pairs (left, right) in hidden_order, which guarantees acyclicity.
    selected_position_pairs: set[tuple[int, int]] = set()
   # Sample without replacement until the number of edges is exactly target_edges.
    while len(selected_position_pairs) < target_edges:
        left, right = rng.sample(range(n), 2)
        # Always place the smaller position first so that only forward edges are generated.
        if left > right:
            left, right = right, left
        selected_position_pairs.add((left, right))

     # Create the random DAG and add all nodes.
    G = nx.DiGraph(name=name or f"random_n{n}_din{mean_indegree:g}")
    G.add_nodes_from(range(n))
    G.add_edges_from((hidden_order[left], hidden_order[right]) for left, right in selected_position_pairs)
    G.graph["hidden_order"] = tuple(hidden_order)
    _validate_dag(G)

    if G.number_of_edges() != target_edges:
        raise AssertionError("随机 DAG 的实际边数与目标边数不一致。")
    return G


def _sample_one_condition_set(G: nx.DiGraph, c_size: int, seed: int) -> tuple[Node, ...]:
    """Current formal protocol: uniformly sample one conditioning set from all nodes."""
    if not 1 <= c_size < G.number_of_nodes():
        raise ValueError("必须满足 1 <= |C| < n。")
    rng = random.Random(seed)
    nodes = tuple(sorted(G.nodes(), key=str))
    return tuple(sorted(rng.sample(nodes, c_size), key=str))

## 2. FGCR Algorithm Code
Algorithm1 constructs the target order $\alpha=(\alpha_C,\alpha_R)$; Algorithm2 copies the graph, performs adjacent swaps and covered-arc reversals, and extracts $H_R=H[R]$.


In [3]:
@dataclass(frozen=True)
class Algorithm1Orders:
    """Store the four orders generated by FGCR Algorithm 1."""
    # beta: the fixed complete topological order of the input DAG.
    beta: tuple[Node, ...]
    # alpha_C: nodes in beta that belong to the conditioning set C, with their relative order preserved.
    alpha_C: tuple[Node, ...]
    # alpha_R: nodes in beta that belong to R=V\C, with their relative order preserved.
    alpha_R: tuple[Node, ...]
    # alpha: the target order, defined as the concatenation of alpha_C and alpha_R.
    alpha: tuple[Node, ...]

@dataclass
class FGCRResult:
    """Store the full graph produced by FGCR Algorithm 2 and the final subgraph over the remaining variables."""
    # H: the full DAG obtained after covered-arc reversals.
    H: nx.DiGraph
    # H_R: the induced subgraph of H on R, which is also the final output used for comparison.
    H_R: nx.DiGraph
    # orders: order information generated by Algorithm 1.
    orders: Algorithm1Orders

def construct_algorithm1_orders(
    G: nx.DiGraph,
    C: Iterable[Node],
    beta: Sequence[Node],
) -> Algorithm1Orders:
    """FGCR Algorithm 1: stably move the nodes in C before those in R within beta."""
    # Convert C to a set so that membership queries take expected O(1) time.
    C_set = set(C)
    # Freeze the input order to prevent the caller from modifying the original list after the function runs.
    beta_tuple = tuple(beta)
    # Select conditioned nodes from beta while preserving their relative order in beta.
    alpha_C = tuple(node for node in beta_tuple if node in C_set)
    # Select remaining nodes from beta while preserving their relative order in beta.
    alpha_R = tuple(node for node in beta_tuple if node not in C_set)
    # Return the complete order object; G is retained to keep the function signature consistent with the algorithm interface.
    return Algorithm1Orders(
        beta=beta_tuple,
        alpha_C=alpha_C,
        alpha_R=alpha_R,
        alpha=alpha_C + alpha_R,
    )


def _cover_and_reverse(H: nx.DiGraph, Y: Node, Z: Node) -> None:
    """Complete the parent sets so that Y->Z becomes a covered arc, then reverse it to Z->Y."""
    # Read the parent set of Y in the current dynamic graph H.
    pa_Y = set(H.predecessors(Y))
    # Read the parent set of Z in the current dynamic graph H.
    pa_Z = set(H.predecessors(Z))
    # If a parent of Y is not yet a parent of Z, add X->Z.
    for X in pa_Y - pa_Z:
        H.add_edge(X, Z)
    # If a parent of Z other than Y is not yet a parent of Y, add X->Y.
    for X in pa_Z - pa_Y - {Y}:
        H.add_edge(X, Y)
    # Remove the original arc Y->Z.
    H.remove_edge(Y, Z)
    # Add the reversed arc Z->Y.
    H.add_edge(Z, Y)


def _construct_full_H(
    G: nx.DiGraph,
    beta: Sequence[Node],
    alpha: Sequence[Node],
) -> nx.DiGraph:
    """Core of FGCR Algorithm 2: transform beta into alpha using adjacent swaps and covered-arc reversals."""
    # beta_current is the current order updated throughout the execution of the algorithm.
    beta_current = list(beta)
    # alpha_list is the fixed target order.
    alpha_list = list(alpha)
    # beta_position maintains node positions in the current order, avoiding repeated calls to list.index().
    beta_position = {node: index for index, node in enumerate(beta_current)}
    # alpha_position stores the fixed position of each node in the target order.
    alpha_position = {node: index for index, node in enumerate(alpha_list)}
    # Algorithm 2 modifies a copy of the original graph so that the input G remains unchanged; copying is included in the timing.
    H = G.copy()

    # Fix the position of each node from right to left according to the target order.
    for Y in reversed(alpha_list):
        # As long as there is a node to the right of Y, check whether Y still needs to move right.
        while beta_position[Y] < len(beta_current) - 1:
            # Obtain the current position of Y.
            current_position = beta_position[Y]
            # Z is the node immediately to the right of Y in the current order.
            Z = beta_current[current_position + 1]
            # If Z should appear after Y in the target order, then Y has reached its correct relative position.
            if alpha_position[Z] > alpha_position[Y]:
                break
            # A covered-arc reversal is required for the adjacent swap only if Y->Z exists in the current graph.
            if H.has_edge(Y, Z):
                _cover_and_reverse(H, Y, Z)
            # Swap Y and Z in the current order.
            beta_current[current_position] = Z
            beta_current[current_position + 1] = Y
            # Update the position indices of both nodes.
            beta_position[Z] = current_position
            beta_position[Y] = current_position + 1
     # Return the full graph H after all order adjustments have been completed.
    return H


def run_fgcr(G: nx.DiGraph, C: Iterable[Node], beta: Sequence[Node]) -> FGCRResult:
    """Untimed interface: run the complete FGCR procedure for validation outside the timed interval."""
    # Run Algorithm 1 to obtain alpha.
    orders = construct_algorithm1_orders(G=G, C=C, beta=beta)
    # Run Algorithm 2 to obtain the full graph H.
    H = _construct_full_H(G=G, beta=orders.beta, alpha=orders.alpha)
    # Extract the induced subgraph of H on R.
    H_R = H.subgraph(orders.alpha_R).copy()
    # Return the complete result.
    return FGCRResult(H=H, H_R=H_R, orders=orders)


def _timed_fgcr_once(
    G: nx.DiGraph,
    C: Iterable[Node],
    beta: Sequence[Node],
) -> tuple[FGCRResult, int, int]:
     """Time one FGCR execution under the strict timing protocol, with Algorithms 1 and 2 timed separately."""
    # Start of Algorithm 1 timing.
    algorithm1_start = time.perf_counter_ns()
    # The timed interval contains only the order construction in Algorithm 1.
    orders = construct_algorithm1_orders(G=G, C=C, beta=beta)
    # End of Algorithm 1 timing.
    algorithm1_end = time.perf_counter_ns()

    # Start of Algorithm 2 timing.
    algorithm2_start = time.perf_counter_ns()
    # Algorithm 2 includes graph copying, adjacent swaps, and covered-arc reversals.
    H = _construct_full_H(G=G, beta=orders.beta, alpha=orders.alpha)
    # Construction of the final H_R is part of the output process of Algorithm 2 and is therefore included in the timed interval.
    H_R = H.subgraph(orders.alpha_R).copy()
    # End of Algorithm 2 timing.
    algorithm2_end = time.perf_counter_ns()

     # Package the result after timing has ended so that dataclass construction is not included in Algorithm 2 time.
    result = FGCRResult(H=H, H_R=H_R, orders=orders)
    # Return the result and the runtimes of Algorithms 1 and 2, all in nanoseconds.
    return result, algorithm1_end - algorithm1_start, algorithm2_end - algorithm2_start


## 3. ROS Algorithm Code
The timed interval for ROS includes copying the graph, removing outgoing edges from conditioned nodes, computing the conditioning-related ancestral region, completing parent sets in reverse topological order, recording fill-in edges, and constructing the final $G_R^\beta$. Input validation and output comparison are both performed outside the timed interval.


In [4]:
@dataclass
class ROSResult:
     """Store the fill-in edge set produced by ROS and the final subgraph over the remaining variables."""
    # E_beta_R: fill-in edges added by ROS with directions determined by beta.
    E_beta_R: tuple[Edge, ...]
    # G_beta_R: the induced subgraph of the original graph on R, augmented with E_beta_R.
    G_beta_R: nx.DiGraph

def _ros_algorithm_core(
    G: nx.DiGraph,
    beta: Sequence[Node],
    C: Iterable[Node],
) -> tuple[list[Edge], nx.DiGraph]:
    """Complete core ROS algorithm; this entire function is included in the ROS timed interval."""
    beta_list = list(beta)
    # Record the position of each node in beta to determine the directions of newly added edges.
    beta_position = {node: index for index, node in enumerate(beta_list)}
    G_work = G.copy()

   # According to the ROS reduction step, remove all edges outgoing from conditioned nodes.
    for conditioned_node in C_set:
        G_work.remove_edges_from(list(G_work.out_edges(conditioned_node)))

    # nodes_of_interest stores the union of An_{G^C}(C) and C.
    nodes_of_interest: set[Node] = set()
    # Compute the ancestors of each conditioned node and take their union.
    for conditioned_node in C_set:
        nodes_of_interest.update(nx.ancestors(G_work, conditioned_node))
        nodes_of_interest.add(conditioned_node)

    # G_0 is the local ancestral subgraph on which ROS performs its search and successive node deletion.
    G_0 = G_work.subgraph(nodes_of_interest).copy()
    # Arrange the local nodes in reverse beta order, consistent with the reverse-order search in ROS.
    vertices = [node for node in reversed(beta_list) if node in nodes_of_interest]
   # Record all fill-in edges added by ROS.
    E_beta_R: list[Edge] = []

   # The final two nodes require no further parent-set completion, so only vertices[:-2] are processed.
    for current_node in vertices[:-2]:
         # Read the parent set of current_node in the current dynamic graph G_0.
        parents = list(G_0.predecessors(current_node))
         # Enumerate all unordered pairs of nodes in the parent set.
        for x, y in combinations(parents, 2):
             # No fill-in edge is needed if the two parent nodes are already adjacent.
            if G_0.has_edge(x, y) or G_0.has_edge(y, x):
                continue
             # The direction of the new edge must be consistent with the input topological order beta.
            if beta_position[x] < beta_position[y]:
                new_edge = (x, y)
            else:
                new_edge = (y, x)
            G_0.add_edge(*new_edge)
            E_beta_R.append(new_edge)
        # Remove the current node from the local graph after it has been processed.
        G_0.remove_node(current_node)

    R_nodes = [node for node in G.nodes if node not in C_set]
    G_beta_R = G.subgraph(R_nodes).copy()
     # Add the fill-in edges found by ROS to obtain the final G_R^beta.
    G_beta_R.add_edges_from(E_beta_R)
    return E_beta_R, G_beta_R


def minimal_fill_edge_set(G: nx.DiGraph, beta: Sequence[Node], C: Iterable[Node]) -> ROSResult:
    """Untimed interface: run the complete ROS procedure for validation outside the timed interval."""
    # Call the core algorithm to obtain the fill-in edges and the final graph.
    fill_edges, G_beta_R = _ros_algorithm_core(G=G, beta=beta, C=C)
    return ROSResult(E_beta_R=tuple(fill_edges), G_beta_R=G_beta_R)


def _timed_ros_once(
    G: nx.DiGraph,
    beta: Sequence[Node],
    C: Iterable[Node],
) -> tuple[ROSResult, int]:
     """Time one execution of the ROS core algorithm under the strict timing protocol."""
    # Start of the ROS timed interval.
    start = time.perf_counter_ns()
    fill_edges, G_beta_R = _ros_algorithm_core(G=G, beta=beta, C=C)
    # End of the ROS timed interval.
    end = time.perf_counter_ns()
    result = ROSResult(E_beta_R=tuple(fill_edges), G_beta_R=G_beta_R)
    return result, end - start


## 4. Structural Diagnostics and Output Consistency Checks Outside the Timed Interval

After all formal timing runs are completed for each $(G,C,\beta)$ instance, the following quantities are computed: 
- $|\operatorname{An}_G(C)\cup C|/n$；
- the number of adjacent swaps performed by FGCR;
- the number of covered-arc reversals;
- the number of covered-arc reversals;
- the maximum parent-set size in the local ancestral region used by ROS;
-whether FGCR's $H_R$  and ROS's $G_R^\beta$ have exactly the same node set and directed edge set. 

In [5]:
def _construct_full_H_with_counts(
    G: nx.DiGraph,
    beta: Sequence[Node],
    alpha: Sequence[Node],
) -> tuple[nx.DiGraph, int, int]:
       """Replicate the control flow of FGCR Algorithm 2 and count swaps and covered-arc reversals outside the timed interval."""
    beta_current = list(beta)
    # Fix the target order.
    alpha_list = list(alpha)
    # Position index for the current order.
    beta_position = {node: index for index, node in enumerate(beta_current)}
    # Position index for the target order.
    alpha_position = {node: index for index, node in enumerate(alpha_list)}
    # Run the diagnostic version of Algorithm 2 on a copy of the original graph.
    H = G.copy()
    # Count the actual number of adjacent swaps.
    swap_count = 0
    # Count the actual number of triggered covered-arc reversals.
    cover_reversal_count = 0

     # Exactly as in Algorithm 2: fix nodes in reverse alpha order.
    for Y in reversed(alpha_list):
        while beta_position[Y] < len(beta_current) - 1:
            current_position = beta_position[Y]
            Z = beta_current[current_position + 1]
            if alpha_position[Z] > alpha_position[Y]:
                break
            # Perform and count a covered-arc reversal only when Y->Z exists.
            if H.has_edge(Y, Z):
                _cover_and_reverse(H, Y, Z)
                cover_reversal_count += 1
             # Complete one adjacent swap and update the positions.
            beta_current[current_position] = Z
            beta_current[current_position + 1] = Y
            beta_position[Z] = current_position
            beta_position[Y] = current_position + 1
            swap_count += 1

    # Return the final full graph and the two counts.
    return H, swap_count, cover_reversal_count


def _ros_region_diagnostics_outside_timing(
    G: nx.DiGraph,
    C: Sequence[Node],
) -> tuple[int, int]:
     """Compute the ROS local-region size k and maximum parent-set size m outside the timed interval."""
    # As in ROS, first copy the graph and remove outgoing edges from conditioned nodes.
    C_set = set(C)
    G_work = G.copy()
    for conditioned_node in C_set:
        G_work.remove_edges_from(list(G_work.out_edges(conditioned_node)))
    # Compute the union of An_{G^C}(C) and C.
    nodes_of_interest: set[Node] = set(C_set)
    for conditioned_node in C_set:
        nodes_of_interest.update(nx.ancestors(G_work, conditioned_node))
    # Compute the maximum number of parents in this local induced subgraph.
    G_0 = G_work.subgraph(nodes_of_interest)
    max_parent_count = max((G_0.in_degree(node) for node in G_0.nodes), default=0)
     # Return k and m.
    return len(nodes_of_interest), int(max_parent_count)


def compute_case_diagnostics_and_verify_outside_timing(
    G: nx.DiGraph,
    C: Sequence[Node],
    beta: Sequence[Node],
    raise_on_mismatch: bool = True,
) -> dict[str, object]:
    """Compute structural diagnostics and verify the final FGCR and ROS outputs outside the timed interval."""
    # Compute the requested union of An_G(C) and C.
    C_set = set(C)
    ancestor_closure = set(C_set)
    for conditioned_node in C_set:
        ancestor_closure.update(nx.ancestors(G, conditioned_node))

    # Construct the FGCR Algorithm 1 orders.
    orders = construct_algorithm1_orders(G=G, C=C_set, beta=beta)
    # Run FGCR Algorithm 2 with counters; this is outside the timed interval.
    H, swap_count, cover_reversal_count = _construct_full_H_with_counts(
        G=G,
        beta=orders.beta,
        alpha=orders.alpha,
    )
    H_R = H.subgraph(orders.alpha_R).copy()
    G_R = G.subgraph(orders.alpha_R).copy()
    fill_edges = set(H_R.edges()) - set(G_R.edges())

    ros_result = minimal_fill_edge_set(G=G, beta=beta, C=C_set)
    fgcr_nodes = set(H_R.nodes())
    ros_nodes = set(ros_result.G_beta_R.nodes())
    fgcr_edges = set(H_R.edges())
    ros_edges = set(ros_result.G_beta_R.edges())
    output_match = fgcr_nodes == ros_nodes and fgcr_edges == ros_edges
    node_difference_count = len(fgcr_nodes.symmetric_difference(ros_nodes))
    edge_difference_count = len(fgcr_edges.symmetric_difference(ros_edges))
    
    # If strict verification is enabled, stop the experiment immediately upon any mismatch.
    if raise_on_mismatch and not output_match:
        raise AssertionError(
            "FGCR 与 ROS 输出不一致："
            f"节点对称差={node_difference_count}，边对称差={edge_difference_count}。"
        )
     # Also record the local-region size k and maximum parent-set size m used in the ROS complexity characterization.
    ros_region_count, ros_max_parent_count = _ros_region_diagnostics_outside_timing(G=G, C=C)

    # Return all diagnostics computed outside the timed interval; Boolean values can be filtered directly in Excel.
    return {
        "ancestor_closure_count": len(ancestor_closure),
        "ancestor_closure_ratio": len(ancestor_closure) / G.number_of_nodes(),
        "ROS_region_count": ros_region_count,
        "ROS_region_ratio": ros_region_count / G.number_of_nodes(),
        "ROS_max_parent_count": ros_max_parent_count,
        "FGCR_swap_count": swap_count,
        "cover_reversal_count": cover_reversal_count,
        "fill_edge_count": len(fill_edges),
        "FGCR_output_n_edges": len(fgcr_edges),
        "ROS_output_n_edges": len(ros_edges),
        "output_match": output_match,
        "node_difference_count": node_difference_count,
        "edge_difference_count": edge_difference_count,
    }

## 5. Paired Timing Function Consistent with the Standard-Network Experiment
FGCR→ROS and ROS→FGCR are alternated according to the instance index and repetition index.

In [6]:
def _run_pair_without_gc(
    G: nx.DiGraph,
    C: Sequence[Node],
    beta: Sequence[Node],
    fgcr_first: bool,
) -> tuple[int, int, int]:
    """Run one pair of algorithms in the specified order with automatic garbage collection disabled."""
    # Record whether garbage collection is enabled before the call so that the original state can be restored afterward.
    gc_was_enabled = gc.isenabled()
    if gc_was_enabled:
        gc.disable()
    try:
        # Alternate which algorithm runs first to control for cache effects and execution-order bias.
        if fgcr_first:
            _, algorithm1_ns, algorithm2_ns = _timed_fgcr_once(G=G, C=C, beta=beta)
            _, ros_total_ns = _timed_ros_once(G=G, beta=beta, C=C)
        else:
            _, ros_total_ns = _timed_ros_once(G=G, beta=beta, C=C)
            _, algorithm1_ns, algorithm2_ns = _timed_fgcr_once(G=G, C=C, beta=beta)
    finally:
        # Restore the garbage-collection state that was active before entering the function, even if an exception occurs.
        if gc_was_enabled:
            gc.enable()
     # Return the two FGCR timing components and the total ROS time, all in nanoseconds.
    return algorithm1_ns, algorithm2_ns, ros_total_ns

def _warm_up_one_case(
    G: nx.DiGraph,
    C: Sequence[Node],
    beta: Sequence[Node],
    repetitions: int,
) -> None:
    """Warm up one complete (G,C,beta) instance; warm-up results are not retained."""
    # Warm up each instance separately, correcting the previous design that warmed up only the first conditioning set.
    for warmup_id in range(1, repetitions + 1):
        gc.collect()
        _run_pair_without_gc(
            G=G,
            C=C,
            beta=beta,
            fgcr_first=warmup_id % 2 == 1,
        )


def run_one_random_runtime_case(
    case_meta: Mapping[str, object],
    G: nx.DiGraph,
    C: Sequence[Node],
    beta: Sequence[Node],
    timing_repetitions: int,
) -> tuple[dict[str, object], list[dict[str, object]]]:
     """Run all formal repetitions for one (G,C,beta) instance and return the median runtimes."""
    # Store FGCR Algorithm 1, Algorithm 2, total FGCR time, and total ROS time separately.
    algorithm1_times_ns: list[int] = []
    algorithm2_times_ns: list[int] = []
    fgcr_total_times_ns: list[int] = []
    ros_total_times_ns: list[int] = []
     # repeat_rows stores the raw timing result from each repetition for verification and re-aggregation.
    repeat_rows: list[dict[str, object]] = []

     # condition_id and order_id jointly determine the execution order of the two algorithms.
    condition_id = int(case_meta["condition_id"])
    order_id = int(case_meta["order_id"])

    # Perform the specified number of formal repetitions for the current instance.
    for repeat_id in range(1, timing_repetitions + 1):
        gc.collect()
        # Use a parity rule to balance the two possible execution orders as evenly as possible.
        fgcr_first = (condition_id + order_id + repeat_id) % 2 == 0
         # Run one pair of algorithms and obtain elapsed times in nanoseconds.
        algorithm1_ns, algorithm2_ns, ros_total_ns = _run_pair_without_gc(
            G=G,
            C=C,
            beta=beta,
            fgcr_first=fgcr_first,
        )
        # Define total FGCR time as the sum of Algorithm 1 and Algorithm 2 runtimes.
        fgcr_total_ns = algorithm1_ns + algorithm2_ns

       # Store the four timing quantities for this repetition.
        algorithm1_times_ns.append(algorithm1_ns)
        algorithm2_times_ns.append(algorithm2_ns)
        fgcr_total_times_ns.append(fgcr_total_ns)
        ros_total_times_ns.append(ros_total_ns)

         # Record this repetition and convert all times consistently to milliseconds.
        repeat_rows.append({
            **case_meta,
            "repeat_id": repeat_id,
            "run_order": "FGCR_then_ROS" if fgcr_first else "ROS_then_FGCR",
            "FGCR_algorithm1_ms": algorithm1_ns / 1_000_000,
            "FGCR_algorithm2_ms": algorithm2_ns / 1_000_000,
            "FGCR_total_ms": fgcr_total_ns / 1_000_000,
            "ROS_total_ms": ros_total_ns / 1_000_000,
        })

     # Take the median independently for each timing quantity; FGCR total time uses the median of the per-repetition sums.
    algorithm1_median_ns = float(median(algorithm1_times_ns))
    algorithm2_median_ns = float(median(algorithm2_times_ns))
    fgcr_total_median_ns = float(median(fgcr_total_times_ns))
    ros_total_median_ns = float(median(ros_total_times_ns))

    # The instance-level record stores median runtimes and the FGCR/ROS runtime ratio.
    case_row = {
        **case_meta,
        "timing_repetitions": timing_repetitions,
        "FGCR_algorithm1_median_ms": algorithm1_median_ns / 1_000_000,
        "FGCR_algorithm2_median_ms": algorithm2_median_ns / 1_000_000,
        "FGCR_total_median_ms": fgcr_total_median_ns / 1_000_000,
        "ROS_total_median_ms": ros_total_median_ns / 1_000_000,
        "FGCR_to_ROS_ratio": fgcr_total_median_ns / ros_total_median_ns,#考虑公平性，时间比为FGCR总时间/ROS时间
    }
    return case_row, repeat_rows


## 6. Main Experiment and Stress Test

The conditioning-set sizes used in the formal experiments remain as follows: a fixed size of $|C|=3$, together with$\lceil0.05n\rceil,\lceil0.10n\rceil,\lceil0.20n\rceil,\lceil0.30n\rceil$. Duplicate sizes are retained only once.

Main experiment: $n\in\{50,100,200,500\}$, average in-degree $d\in\{2,4,8\}$, with 10 random graphs generated for each configuration.

Stress test:$n=1000$、average in-degree $d\in\{2,4\}$, with 5 random graphs generated for each configuration. The stress-test results are saved separately.


In [7]:
@dataclass(frozen=True)
class RandomRuntimeConfig:
    """Store a set of random-network timing experiment parameters in one place."""
    # experiment_label distinguishes the main experiment, stress test, and smoke test.
    experiment_label: str = "main"
    ns: tuple[int, ...] = (50, 100, 200, 500)
    # mean_indegrees: target average in-degree d=|E|/n.
    mean_indegrees: tuple[float, ...] = (2.0, 4.0, 8.0)
    # fixed_c_size: the current formal design additionally retains a fixed conditioning-set size of 3.
    fixed_c_size: int = 3
    c_ratios: tuple[float, ...] = (0.05, 0.10, 0.20, 0.30)
    # graphs_per_cell: number of independent random DAGs generated for each (n,d) parameter cell.
    graphs_per_cell: int = 10
    # n_fixed_orders: number of distinct complete topological orders fixed for each graph.
    n_fixed_orders: int = 4
    # warmup_repetitions: number of warm-up runs performed separately for each (G,C,beta) instance.
    warmup_repetitions: int = 3
    # timing_repetitions: number of formal timing repetitions for each instance.
    timing_repetitions: int = 10
    graph_base_seed: int = 20260731
    condition_seed_offset: int = 1_000_000_000_000
    order_seed_offset: int = 2_000_000_000_000
    # If raise_on_output_mismatch=True, stop immediately when FGCR and ROS outputs differ.
    raise_on_output_mismatch: bool = True

def _validate_random_runtime_config(config: RandomRuntimeConfig) -> None:
    """Validate the experiment configuration once before the formal experiment begins."""
    if not config.ns or any(n < 2 for n in config.ns):
        raise ValueError("ns 必须由至少为2的节点数组成。")
    if not config.mean_indegrees or any(d <= 0 for d in config.mean_indegrees):
        raise ValueError("mean_indegrees 必须由正数组成。")
    if not 1 <= config.fixed_c_size < min(config.ns):
        raise ValueError("fixed_c_size 必须满足 1 <= fixed_c_size < min(ns)。")
    if not config.c_ratios or any(not 0 < ratio < 1 for ratio in config.c_ratios):
        raise ValueError("c_ratios 中每个比例必须严格位于0和1之间。")
    if config.graphs_per_cell <= 0:
        raise ValueError("graphs_per_cell 必须为正整数。")
    if config.n_fixed_orders <= 0:
        raise ValueError("n_fixed_orders 必须为正整数。")
    if config.warmup_repetitions < 0:
        raise ValueError("warmup_repetitions 不能为负数。")
    if config.timing_repetitions <= 0:
        raise ValueError("timing_repetitions 必须为正整数。")
    for n in config.ns:
        for d in config.mean_indegrees:
            if int(round(n * d)) > n * (n - 1) // 2:
                raise ValueError(f"n={n}, d={d} 对应的目标边数超过简单 DAG 上限。")


def _condition_size_specs(n: int, config: RandomRuntimeConfig) -> list[dict[str, object]]:
    """Formal design: fixed size 3 plus four ratios, with duplicate sizes removed."""
    # First add the fixed conditioning-set size.
    specs: list[dict[str, object]] = [{
        "condition_id": 1,
        "condition_design": f"fixed_{config.fixed_c_size}",
        "nominal_c_ratio": None,
        "c_size": config.fixed_c_size,
        "actual_c_ratio": config.fixed_c_size / n,
    }]
     # seen_sizes prevents duplication between the fixed size and ratio-based sizes.
    seen_sizes = {config.fixed_c_size}

    # Generate conditioning-set sizes ceil(ratio*n) for each ratio.
    for ratio in config.c_ratios:
        c_size = int(math.ceil(ratio * n))
        if c_size in seen_sizes:
            continue
        seen_sizes.add(c_size)
        specs.append({
            "condition_id": len(specs) + 1,
            "condition_design": f"ratio_{ratio:.2f}",
            "nominal_c_ratio": ratio,
            "c_size": c_size,
            "actual_c_ratio": c_size / n,
        })
     # Return all distinct conditioning-set sizes for this n.
    return specs

# Main-experiment configuration.
MAIN_RANDOM_RUNTIME_CONFIG = RandomRuntimeConfig(
    experiment_label="main",
    ns=(50, 100, 200, 500),
    mean_indegrees=(2.0, 4.0, 8.0),
    graphs_per_cell=10,
)

# Stress-test configuration.
STRESS_RANDOM_RUNTIME_CONFIG = RandomRuntimeConfig(
    experiment_label="stress_n1000",
    ns=(1000,),
    mean_indegrees=(2.0,4.0),
    graphs_per_cell=5,
)

FORMAL_RANDOM_RUNTIME_CONFIG = MAIN_RANDOM_RUNTIME_CONFIG

# The smoke test is used to check the workflow, output columns, and consistency verification.
SMOKE_RANDOM_RUNTIME_CONFIG = RandomRuntimeConfig(
    experiment_label="smoke",
    ns=(50,),
    mean_indegrees=(2.0,),
    fixed_c_size=3,
    c_ratios=(0.10,),
    graphs_per_cell=1,
    n_fixed_orders=2,
    warmup_repetitions=1,
    timing_repetitions=2,
)


def show_parameter_grid(
    config: RandomRuntimeConfig = MAIN_RANDOM_RUNTIME_CONFIG,
) -> pd.DataFrame:
    """Display the node sizes, densities, and conditioning-set sizes expanded from a configuration."""
    # Each row corresponds to one (n,d,conditioning-set size) parameter cell.
    rows: list[dict[str, object]] = []
    for n in config.ns:
        for mean_indegree in config.mean_indegrees:
            for spec in _condition_size_specs(n, config):
                rows.append({
                    "experiment_label": config.experiment_label,
                    "n": n,
                    "mean_indegree_target": mean_indegree,
                    "condition_design": spec["condition_design"],
                    "c_size": spec["c_size"],
                    "actual_c_ratio": spec["actual_c_ratio"],
                })
    return pd.DataFrame(rows)

show_parameter_grid(MAIN_RANDOM_RUNTIME_CONFIG)

,experiment_label,n,mean_indegree_target,condition_design,c_size,actual_c_ratio
0,main,50,2.0,fixed_3,3,0.060
1,main,50,2.0,ratio_0.10,5,0.100
2,main,50,2.0,ratio_0.20,10,0.200
3,main,50,2.0,ratio_0.30,15,0.300
4,main,50,4.0,fixed_3,3,0.060
5,main,50,4.0,ratio_0.10,5,0.100
6,main,50,4.0,ratio_0.20,10,0.200
7,main,50,4.0,ratio_0.30,15,0.300
8,main,50,8.0,fixed_3,3,0.060
9,main,50,8.0,ratio_0.10,5,0.100


## 7. Main Experimental Loop: Per-Instance Warm-Up and Timing, Followed by Per-Instance Verification Outside the Timed Interval

The processing sequence is:
$
\text{Generate }G,C,\beta
\rightarrow
\text{warm up the current }(G,C,\beta)\text{ instance}
\rightarrow
\text{perform formal paired timing}
\rightarrow
\text{complete all timing runs for the current graph}
\rightarrow
\text{perform structural diagnostics and output-consistency verification}.
$

In [8]:
def _graph_seed(config: RandomRuntimeConfig, n: int, mean_indegree: float, graph_id: int) -> int:
    """Deterministically map the configuration, node count, average in-degree, and graph ID to a random seed."""
    degree_code = int(round(mean_indegree * 100))
    # Different combinations of n, d, and graph_id are assigned to non-overlapping seed ranges.
    return config.graph_base_seed + n * 10_000_000 + degree_code * 10_000 + graph_id


def _summarize_random_runtime(case_details: pd.DataFrame) -> pd.DataFrame:
    """Summarize instance-level median runtimes and structural diagnostics by parameter cell."""
    # These columns jointly define an experimental parameter cell.
    group_columns = [
        "experiment_label",
        "n_nodes",
        "mean_indegree_target",
        "n_edges",
        "mean_indegree_actual",
        "condition_design",
        "c_size",
        "actual_c_ratio",
    ]
    rows: list[dict[str, object]] = []

   # Summarize each parameter cell independently.
    for keys, group in case_details.groupby(group_columns, sort=False, dropna=False):
        key_values = dict(zip(group_columns, keys))
        # Average the instance-level median runtimes
        fgcr_algorithm1_mean = float(group["FGCR_algorithm1_median_ms"].mean())
        fgcr_algorithm2_mean = float(group["FGCR_algorithm2_median_ms"].mean())
        fgcr_total_mean = float(group["FGCR_total_median_ms"].mean())
        ros_total_mean = float(group["ROS_total_median_ms"].mean())

        # First average across the four topological orders within each graph, then compute the graph-level FGCR/ROS runtime ratio.
        graph_ratios: list[float] = []
        for _, graph_group in group.groupby("graph_seed", sort=False):
                graph_fgcr = float(graph_group["FGCR_total_median_ms"].mean())
                graph_ros = float(graph_group["ROS_total_median_ms"].mean())
                graph_ratios.append(graph_fgcr / graph_ros)

        # Compute the geometric mean and approximate 95% interval on the log scale.
        log_ratios = [math.log(ratio) for ratio in graph_ratios]
        log_mean = sum(log_ratios) / len(log_ratios)
        graph_ratio_geomean = math.exp(log_mean)
        if len(log_ratios) >= 2:
            variance = sum((value - log_mean) ** 2 for value in log_ratios) / (len(log_ratios) - 1)
            standard_error = math.sqrt(variance / len(log_ratios))
            ratio_ci95_low = math.exp(log_mean - 1.96 * standard_error)
            ratio_ci95_high = math.exp(log_mean + 1.96 * standard_error)
        else:
            ratio_ci95_low = float("nan")
            ratio_ci95_high = float("nan")

         # Store the summary statistics for the current parameter cell.
        rows.append({
            **key_values,
            "n_graphs": int(group["graph_seed"].nunique()),
            "n_orders": int(group["order_id"].nunique()),
            "n_instances": len(group),
            "FGCR_algorithm1_mean_ms": fgcr_algorithm1_mean,
            "FGCR_algorithm2_mean_ms": fgcr_algorithm2_mean,
            "FGCR_total_mean_ms": fgcr_total_mean,
            "ROS_total_mean_ms": ros_total_mean,
            "FGCR_to_ROS_ratio_of_means": fgcr_total_mean / ros_total_mean,
            "graph_ratio_geomean": graph_ratio_geomean,
            "graph_ratio_ci95_low": ratio_ci95_low,
            "graph_ratio_ci95_high": ratio_ci95_high,
            "ancestor_closure_ratio_mean": float(group["ancestor_closure_ratio"].mean()),
            "ROS_region_ratio_mean": float(group["ROS_region_ratio"].mean()),
            "ROS_max_parent_count_mean": float(group["ROS_max_parent_count"].mean()),
            "FGCR_swap_count_mean": float(group["FGCR_swap_count"].mean()),
            "cover_reversal_count_mean": float(group["cover_reversal_count"].mean()),
            "fill_edge_count_mean": float(group["fill_edge_count"].mean()),
            "output_match_rate": float(group["output_match"].mean()),
            "edge_difference_count_sum": int(group["edge_difference_count"].sum()),
        })

    # Return the parameter-cell summary table.
    return pd.DataFrame(rows)


def run_random_runtime_experiment(
    config: RandomRuntimeConfig = MAIN_RANDOM_RUNTIME_CONFIG,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run one main-experiment or stress-test configuration without writing output files."""
    # Validate all configuration parameters before the formal experiment begins.
    _validate_random_runtime_config(config)

    case_rows: list[dict[str, object]] = []
    repeat_rows: list[dict[str, object]] = []
    graph_input_rows: list[dict[str, object]] = []
    condition_input_rows: list[dict[str, object]] = []
    fixed_order_rows: list[dict[str, object]] = []

    # Iterate over node counts.
    for n in config.ns:
        condition_specs = _condition_size_specs(n, config)

         # Iterate over target average in-degree d=|E|/n.
        for mean_indegree in config.mean_indegrees:
            for graph_id in range(1, config.graphs_per_cell + 1):
                graph_wall_start = time.perf_counter()
                graph_seed = _graph_seed(config, n, mean_indegree, graph_id)
                graph_name = f"random_n{n}_din{mean_indegree:g}_g{graph_id:02d}"
                G = generate_random_dag_exact_edges(
                    n=n,
                    mean_indegree=mean_indegree,
                    seed=graph_seed,
                    name=graph_name,
                )
                actual_mean_indegree = G.number_of_edges() / G.number_of_nodes()

                # Store graph-level input information; the graph itself can be reconstructed from the seed and parameters.
                graph_input_rows.append({
                    "experiment_label": config.experiment_label,
                    "graph_name": graph_name,
                    "graph_id": graph_id,
                    "graph_seed": graph_seed,
                    "n_nodes": G.number_of_nodes(),
                    "n_edges": G.number_of_edges(),
                    "mean_indegree_target": mean_indegree,
                    "mean_indegree_actual": actual_mean_indegree,
                })

                 # Pre-generate all conditioning sets for the current graph; generation is outside the algorithm timing.
                prepared_conditions: list[dict[str, object]] = []
                for spec in condition_specs:
                     # Pre-generate all conditioning sets for the current graph; generation is outside the algorithm timing.
                    condition_id = int(spec["condition_id"])
                    condition_seed = config.condition_seed_offset + graph_seed * 100 + condition_id
                    # Current formal design: uniformly sample one conditioning set from all nodes.
                    C = _sample_one_condition_set(
                        G=G,
                        c_size=int(spec["c_size"]),
                        seed=condition_seed,
                    )
                    # Store the conditioning-set object for subsequent warm-up, timing, and diagnostics.
                    prepared = {**spec, "condition_seed": condition_seed, "C": C}
                    prepared_conditions.append(prepared)
                    # Store the conditioning-set input record.
                    condition_input_rows.append({
                        "experiment_label": config.experiment_label,
                        "graph_name": graph_name,
                        "graph_id": graph_id,
                        "graph_seed": graph_seed,
                        "n_nodes": G.number_of_nodes(),
                        "mean_indegree_target": mean_indegree,
                        "condition_id": condition_id,
                        "condition_design": spec["condition_design"],
                        "condition_selection": "global_random",
                        "nominal_c_ratio": spec["nominal_c_ratio"],
                        "c_size": spec["c_size"],
                        "actual_c_ratio": spec["actual_c_ratio"],
                        "condition_seed": condition_seed,
                        "C": _encode_nodes(C),
                    })

                # Generate a fixed number of distinct complete topological orders for the current graph.
                order_seed_base = config.order_seed_offset + graph_seed * 100
                fixed_orders = generate_fixed_random_topological_orders(
                    G=G,
                    n_orders=config.n_fixed_orders,
                    seed_base=order_seed_base,
                )
                # Store each topological-order input.
                for order_info in fixed_orders:
                    fixed_order_rows.append({
                        "experiment_label": config.experiment_label,
                        "graph_name": graph_name,
                        "graph_id": graph_id,
                        "graph_seed": graph_seed,
                        "n_nodes": G.number_of_nodes(),
                        "mean_indegree_target": mean_indegree,
                        "order_id": order_info["order_id"],
                        "beta_seed": order_info["beta_seed"],
                        "beta_length": len(order_info["beta"]),
                        "beta": _encode_order(order_info["beta"]),
                    })

               # current_graph_cases temporarily stores all instances of the current graph
                # so that diagnostics can be performed only after all timing runs are complete.
                current_graph_cases: list[
                    tuple[dict[str, object], Sequence[Node], Sequence[Node]]
                ] = []

                 # Each fixed topological order and each conditioning set form one paired timing instance.
                for order_info in fixed_orders:
                    order_id = int(order_info["order_id"])
                    beta_seed = int(order_info["beta_seed"])
                    beta = tuple(order_info["beta"])

                    #  Warm up each (G,C,beta) instance separately, then immediately perform its formal timing runs.
                    for prepared in prepared_conditions:
                        _warm_up_one_case(
                            G=G,
                            C=prepared["C"],
                            beta=beta,
                            repetitions=config.warmup_repetitions,
                        )

                        # Construct fixed metadata for the current instance;these operations are outside the algorithm timing.
                        case_meta = {
                            "experiment_label": config.experiment_label,
                            "graph_name": graph_name,
                            "graph_id": graph_id,
                            "graph_seed": graph_seed,
                            "n_nodes": G.number_of_nodes(),
                            "n_edges": G.number_of_edges(),
                            "mean_indegree_target": mean_indegree,
                            "mean_indegree_actual": actual_mean_indegree,
                            "condition_id": prepared["condition_id"],
                            "condition_design": prepared["condition_design"],
                            "condition_selection": prepared.get("condition_selection", "global_random"),
                            "nominal_c_ratio": prepared["nominal_c_ratio"],
                            "c_size": prepared["c_size"],
                            "actual_c_ratio": prepared["actual_c_ratio"],
                            "condition_seed": prepared["condition_seed"],
                            "order_id": order_id,
                            "beta_seed": beta_seed,
                        }
                         # Perform all formal timing repetitions for the current instance.
                        case_row, current_repeat_rows = run_one_random_runtime_case(
                            case_meta=case_meta,
                            G=G,
                            C=prepared["C"],
                            beta=beta,
                            timing_repetitions=config.timing_repetitions,
                        )
                        case_rows.append(case_row)
                        repeat_rows.extend(current_repeat_rows)
                        current_graph_cases.append((case_row, prepared["C"], beta))

                for case_row, C, beta in current_graph_cases:
                    diagnostics = compute_case_diagnostics_and_verify_outside_timing(
                        G=G,
                        C=C,
                        beta=beta,
                        raise_on_mismatch=config.raise_on_output_mismatch,
                    )
                    case_row.update(diagnostics)
                 # Compute the total wall-clock time for the current graph and report progress.
                graph_wall_seconds = time.perf_counter() - graph_wall_start
                print(
                    f"[{config.experiment_label}] n={n}, mean_indegree={mean_indegree:g}, "
                    f"graph={graph_id}/{config.graphs_per_cell} 完成；"
                    f"实例={len(current_graph_cases)}，墙钟时间={graph_wall_seconds:.1f}s"
                )

     # Convert all record collections to DataFrames.
    case_details = pd.DataFrame(case_rows)
    raw_repeat_times = pd.DataFrame(repeat_rows)
    graph_inputs = pd.DataFrame(graph_input_rows)
    condition_inputs = pd.DataFrame(condition_input_rows)
    fixed_orders_df = pd.DataFrame(fixed_order_rows)
    cell_summary = _summarize_random_runtime(case_details)

   # Dynamically compute the expected number of records under the current configuration.
    expected_graphs = len(config.ns) * len(config.mean_indegrees) * config.graphs_per_cell
    expected_conditions = sum(
        len(_condition_size_specs(n, config))
        for n in config.ns
        for _ in config.mean_indegrees
    ) * config.graphs_per_cell
    expected_orders = expected_graphs * config.n_fixed_orders
    expected_cases = expected_conditions * config.n_fixed_orders
    expected_repeat_rows = expected_cases * config.timing_repetitions

     # Verify each record count to prevent missing or duplicated loop levels.
    if len(graph_inputs) != expected_graphs:
        raise AssertionError(f"随机图数量错误：得到 {len(graph_inputs)}，预期 {expected_graphs}。")
    if len(condition_inputs) != expected_conditions:
        raise AssertionError(f"条件集输入数量错误：得到 {len(condition_inputs)}，预期 {expected_conditions}。")
    if len(fixed_orders_df) != expected_orders:
        raise AssertionError(f"固定拓扑序数量错误：得到 {len(fixed_orders_df)}，预期 {expected_orders}。")
    if len(case_details) != expected_cases:
        raise AssertionError(f"计时实例数量错误：得到 {len(case_details)}，预期 {expected_cases}。")
    if len(raw_repeat_times) != expected_repeat_rows:
        raise AssertionError(
            f"逐次计时记录数量错误：得到 {len(raw_repeat_times)}，预期 {expected_repeat_rows}。"
        )
    if config.raise_on_output_mismatch and not bool(case_details["output_match"].all()):
        raise AssertionError("存在未通过 FGCR/ROS 输出一致性检查的实例。")

    return cell_summary, case_details, raw_repeat_times, graph_inputs, condition_inputs, fixed_orders_df


## 8. Saving Results and Separate Execution Entry Points
The main experiment and the $n=1000$ stress test are saved separately as two Excel files. Each file contains parameter-cell summaries, instance-level median runtimes and diagnostics computed outside the timed interval, raw repetition-level runtimes, random-graph inputs, conditioning-set inputs, fixed topological orders, and the complete experimental configuration.


In [9]:
# All random-network results are written to this directory.
RANDOM_RESULT_DIR = Path("./results_random_networks")
MAIN_RANDOM_RUNTIME_RESULT_FILE = RANDOM_RESULT_DIR / "FGCR_ROS_random_runtime_main.xlsx"
STRESS_RANDOM_RUNTIME_RESULT_FILE = RANDOM_RESULT_DIR / "FGCR_ROS_random_runtime_stress_n1000.xlsx"


def save_random_runtime_results(
    cell_summary: pd.DataFrame,
    case_details: pd.DataFrame,
    raw_repeat_times: pd.DataFrame,
    graph_inputs: pd.DataFrame,
    condition_inputs: pd.DataFrame,
    fixed_orders: pd.DataFrame,
    config: RandomRuntimeConfig,
    output_file: str | Path,
) -> Path:
    """Write all results from one experiment configuration to a single Excel workbook."""
    # Convert the output path to a Path object.
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    config_dict = asdict(config)
    config_rows: list[dict[str, object]] = []
    for parameter, value in config_dict.items():
        config_rows.append({
            "parameter": parameter,
            "value": json.dumps(value, ensure_ascii=False) if isinstance(value, tuple) else value,
        })
    config_df = pd.DataFrame(config_rows)

     # Create the Excel workbook and write all seven worksheets in one pass.
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        cell_summary.to_excel(writer, sheet_name="cell_summary", index=False)
        case_details.to_excel(writer, sheet_name="case_details", index=False)
        raw_repeat_times.to_excel(writer, sheet_name="raw_repeat_times", index=False)
        graph_inputs.to_excel(writer, sheet_name="graph_inputs", index=False)
        condition_inputs.to_excel(writer, sheet_name="condition_inputs", index=False)
        fixed_orders.to_excel(writer, sheet_name="fixed_orders", index=False)
        config_df.to_excel(writer, sheet_name="config", index=False)
    return output_file


def run_random_runtime_only(
    config: RandomRuntimeConfig = MAIN_RANDOM_RUNTIME_CONFIG,
    output_file: str | Path = MAIN_RANDOM_RUNTIME_RESULT_FILE,
) -> dict[str, object]:
     """Run one experiment configuration independently, save the results, and return all result tables in memory."""
    # Run the main experimental loop and obtain the six result tables.
    (
        cell_summary,
        case_details,
        raw_repeat_times,
        graph_inputs,
        condition_inputs,
        fixed_orders,
    ) = run_random_runtime_experiment(config=config)

    saved_path = save_random_runtime_results(
        cell_summary=cell_summary,
        case_details=case_details,
        raw_repeat_times=raw_repeat_times,
        graph_inputs=graph_inputs,
        condition_inputs=condition_inputs,
        fixed_orders=fixed_orders,
        config=config,
        output_file=output_file,
    )

     # Print the parameter-cell summary and key record counts to the console.
    print(f"\n{config.experiment_label} 参数格计时汇总：")
    print(cell_summary.to_string(index=False))
    print(f"\n随机图数量：{len(graph_inputs)}")
    print(f"正式计时实例数：{len(case_details)}")
    print(f"原始重复计时记录数：{len(raw_repeat_times)}")
    print(f"输出一致实例数：{int(case_details['output_match'].sum())}/{len(case_details)}")
    print(f"结果文件：{saved_path.resolve()}")

     # Return the result dictionary for subsequent analysis and plotting in the Notebook.
    return {
        "cell_summary": cell_summary,
        "case_details": case_details,
        "raw_repeat_times": raw_repeat_times,
        "graph_inputs": graph_inputs,
        "condition_inputs": condition_inputs,
        "fixed_orders": fixed_orders,
        "output_file": saved_path,
    }


## 9. Formal Experiment Execution Cell

In [ ]:
# 1. Main experiment: n={50,100,200,500}, d={2,4,8}, with 10 graphs per cell.
main_runtime_result = run_random_runtime_only(
    config=MAIN_RANDOM_RUNTIME_CONFIG,
    output_file=MAIN_RANDOM_RUNTIME_RESULT_FILE, )

[main] n=50, mean_indegree=2, graph=1/10 完成；实例=16，墙钟时间=9.6s
[main] n=50, mean_indegree=2, graph=2/10 完成；实例=16，墙钟时间=10.1s
[main] n=50, mean_indegree=2, graph=3/10 完成；实例=16，墙钟时间=9.5s
[main] n=50, mean_indegree=2, graph=4/10 完成；实例=16，墙钟时间=9.2s
[main] n=50, mean_indegree=2, graph=5/10 完成；实例=16，墙钟时间=8.7s
[main] n=50, mean_indegree=2, graph=6/10 完成；实例=16，墙钟时间=8.3s
[main] n=50, mean_indegree=2, graph=7/10 完成；实例=16，墙钟时间=8.3s
[main] n=50, mean_indegree=2, graph=8/10 完成；实例=16，墙钟时间=8.1s
[main] n=50, mean_indegree=2, graph=9/10 完成；实例=16，墙钟时间=8.6s
[main] n=50, mean_indegree=2, graph=10/10 完成；实例=16，墙钟时间=8.9s
[main] n=50, mean_indegree=4, graph=1/10 完成；实例=16，墙钟时间=8.5s
[main] n=50, mean_indegree=4, graph=2/10 完成；实例=16，墙钟时间=8.5s
[main] n=50, mean_indegree=4, graph=3/10 完成；实例=16，墙钟时间=9.6s
[main] n=50, mean_indegree=4, graph=4/10 完成；实例=16，墙钟时间=9.1s
[main] n=50, mean_indegree=4, graph=5/10 完成；实例=16，墙钟时间=9.1s
[main] n=50, mean_indegree=4, graph=6/10 完成；实例=16，墙钟时间=8.9s
[main] n=50, mean_indegree=4, graph=7/

In [11]:
# 2. Independent stress test: n=1000, d={2,4}, with 5 graphs per cell.
stress_runtime_result = run_random_runtime_only(
    config=STRESS_RANDOM_RUNTIME_CONFIG,
    output_file=STRESS_RANDOM_RUNTIME_RESULT_FILE,
)


[stress_n1000] n=1000, mean_indegree=2, graph=1/5 完成；实例=20，墙钟时间=148.4s
[stress_n1000] n=1000, mean_indegree=2, graph=2/5 完成；实例=20，墙钟时间=174.0s
[stress_n1000] n=1000, mean_indegree=2, graph=3/5 完成；实例=20，墙钟时间=152.0s
[stress_n1000] n=1000, mean_indegree=2, graph=4/5 完成；实例=20，墙钟时间=149.5s
[stress_n1000] n=1000, mean_indegree=2, graph=5/5 完成；实例=20，墙钟时间=140.7s
[stress_n1000] n=1000, mean_indegree=4, graph=1/5 完成；实例=20，墙钟时间=849.9s
[stress_n1000] n=1000, mean_indegree=4, graph=2/5 完成；实例=20，墙钟时间=972.2s
[stress_n1000] n=1000, mean_indegree=4, graph=3/5 完成；实例=20，墙钟时间=787.1s
[stress_n1000] n=1000, mean_indegree=4, graph=4/5 完成；实例=20，墙钟时间=902.1s
[stress_n1000] n=1000, mean_indegree=4, graph=5/5 完成；实例=20，墙钟时间=940.8s

stress_n1000 参数格计时汇总：
experiment_label  n_nodes  mean_indegree_target  n_edges  mean_indegree_actual condition_design  c_size  actual_c_ratio  n_graphs  n_orders  n_instances  FGCR_algorithm1_mean_ms  FGCR_algorithm2_mean_ms  FGCR_total_mean_ms  ROS_total_mean_ms  FGCR_to_ROS_ratio_of_mea